# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux

from skewerjax.optical_depth import line_profile

from skewerjax.constants.physical_constants import phys
from skewerjax.optical_depth.voigt import Lorentzian

# Validation against analytical result with $T=0,\, v_{LoS}$=0, a=0.9995

### Define the fields to sample

In [ ]:
L_box = 1.0 # in Mpc

T0 = jnp.finfo(jnp.dtype('float32')).tiny
v0 = 0.0

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0, T0_max=T0, L_box=L_box)), # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),       # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                          # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))                         # UNITS: [GHz]
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0, T0_max=T0, L_box=L_box)), # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),       # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                          # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))                         # UNITS: [GHz]
    }
})

### Define wavelength range and lines to study

In [ ]:
wavelengths = jnp.linspace(970, 1230, 2**20)  # in Angstroms
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

### Define points to evaluate

In [ ]:
quadri_vectors = jnp.array([0.9995, L_box/2, L_box/2, L_box/2])[None, :] # single points at the center of the box, with scale_factor (first entry) = 1.0
LoS_directions = jnp.array([[0.0, 0.0, 1.0]])                         # line of sight direction is along the z-axis
scale_factor_obs = jnp.ones(quadri_vectors.shape[0:-1])               # scale factor wrt the observer, here the observer is at the center of the box, a0=1

### Compute line profile

In [ ]:
phi_ns = line_profile.compute_line_profiles(
    quadri_vectors,
    LoS_directions,
    scale_factor_obs,
    wavelengths,
    line_bundle,
    fields,
    voigt_implementation_mode = "ExoJAX"
)
print("Shape of phi_ns", phi_ns.shape, ". Expected shape (N_lines, *batch_shape, N_lambda) = (",line_bundle.n_lines,",",quadri_vectors.shape[:-1],",",len(wavelengths),")")
phi_single_point = phi_ns[:, 0] # in here we are only studing a single point, *batch_shape = (1,), for this reason we take the first element

### Analytic benchmark

In [ ]:
gamma_GHz = line_bundle.A21_GHz / (4.0 * jnp.pi)

lambdas_A_prime = line_profile.compute_lambda_prime(
    wavelengths,
    scale_factor_obs=scale_factor_obs,
    scale_factor=quadri_vectors[:, 0],
    LoS_velocity=jnp.zeros(line_bundle.n_lines)[:, None]
)[:, 0]

lambda0_A = jnp.asarray(line_bundle.lambda0_A)  # (N_lines,)
nu_GHz = ((1.0 / lambdas_A_prime) - (1.0 / lambda0_A[:, None])) * phys.c * 10.0

phi_analytic = Lorentzian(nu_GHz, gamma_GHz[:, None])

### Validation plot

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)           # (N_lambda,)
phi_np = np.asarray(phi_single_point)          # (N_lines, N_lambda)
phi_analytic = np.asarray(phi_analytic)        # (N_lines, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:line_bundle.n_lines])

fig, ax = plt.subplots(figsize=(14, 6))

for i_sp, line in enumerate(line_bundle.lines):
    ax.scatter(
        np.array([line_bundle.lines[i_sp].lambda0_A * (scale_factor_obs[0]/quadri_vectors[0, 0]) ]), 
        np.array([4./line_bundle.lines[i_sp].A21_GHz]), 
        marker="x", color=colors[i_sp], s=100, alpha=1.0
    )
    ax.axvline(line.lambda0_A, color=colors[i_sp], linestyle="-", linewidth=1.5, alpha=1.0)
    ax.axvline(line.lambda0_A * (scale_factor_obs[0]/quadri_vectors[0, 0]), color=colors[i_sp], linestyle="--", linewidth=1.5, alpha=1.0)
    ax.plot(lambdas_np, phi_np[i_sp, :], color=colors[i_sp], linestyle='-', linewidth=2.0)
    ax.plot(lambdas_np, phi_analytic[i_sp, :], color=colors[i_sp], linestyle=':', linewidth=4.0, alpha=0.6)

ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

handles = [
    Line2D([0], [0], color='grey', lw=4, linestyle='-', label='SkewerJAX'),
    Line2D([0], [0], color='grey', lw=4, linestyle=':', label='Analytical'),
    Line2D([0], [0], color='grey', marker='x', linestyle='None', markersize=10, 
            label=r'Peak value $\frac{4}{A_{21}}$')
]
legend = ax.legend(handles=handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(legend)

handles = [
    Line2D([0], [0], color='grey', lw=4, linestyle='-.', label='rest-frame absorption wavelength'),
    Line2D([0], [0], color='grey', lw=4, linestyle='--', label='redshifted absorption wavelength')
]
legend = ax.legend(handles=handles, loc="lower left", fontsize=16, ncols=1)
ax.add_artist(legend)


handles = []
for i_sp, line in enumerate(line_bundle.lines):
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    handles.append(Line2D([0], [0], color=colors[i_sp], lw=3, linestyle='-', label=lbl))
leg_L = ax.legend(handles=handles, loc="lower right", fontsize=16, ncol=1)

fig.tight_layout()
plt.show()





# Zoom around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
N_lines = line_bundle.n_lines

fig, axes = plt.subplots(
    nrows=2, ncols=N_lines, 
    figsize=(14, 8), 
    sharex='col',
    sharey='row',
    gridspec_kw=dict(height_ratios=[2, 1])
)

if N_lines == 1:
    axes = axes.reshape(2, 1)

for i_sp, (ax_main, ax_res, line) in enumerate(zip(axes[0], axes[1], line_bundle.lines)):
    lambda0 = line.lambda0_A
    
    ax_main.scatter(
        np.array([lambda0 * (scale_factor_obs[0]/quadri_vectors[0, 0])]), 
        np.array([4./line_bundle.lines[i_sp].A21_GHz]), 
        marker="x", color=colors[i_sp], s=100, alpha=1.0
    )
    ax_main.axvline(lambda0, color=colors[i_sp], linestyle="-.", linewidth=1.5)
    ax_main.axvline(line.lambda0_A * (scale_factor_obs[0]/quadri_vectors[0, 0]), color=colors[i_sp], linestyle="--", linewidth=1.5, alpha=1.0)
    ax_res.axvline(lambda0, color=colors[i_sp], linestyle="-.", linewidth=1.5)
    ax_res.axvline(line.lambda0_A * (scale_factor_obs[0]/quadri_vectors[0, 0]), color=colors[i_sp], linestyle="--", linewidth=1.5, alpha=1.0)
    
    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]
    
    ax_main.plot(lam_win, phi_np[i_sp, mask], color=colors[i_sp], linestyle='-', linewidth=2.0, label='SkewerJAX')
    ax_main.plot(lam_win, phi_analytic[i_sp, mask], color=colors[i_sp], linestyle=':', linewidth=4.0, alpha=0.6, label='Analytical')
    
    ax_main.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax_main.set_yscale("log")
    ax_main.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
    
    residual = phi_np[i_sp, mask] / phi_analytic[i_sp, mask] - 1
    
    ax_res.plot(lam_win, residual, color=colors[i_sp], linestyle='-', linewidth=1.0)
    ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
    ax_res.fill_between(lam_win, -1e-6, 1e-6, color="grey", alpha=0.2)
    ax_res.hlines(1e-6, xmin=lam_win[0], xmax=lam_win[-1], color="k", linewidth=1, linestyle='--', alpha=0.5)
    ax_res.hlines(-1e-6, xmin=lam_win[0], xmax=lam_win[-1], color="k", linewidth=1, linestyle='--', alpha=0.5)
    
    ax_res.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax_res.set_ylabel("residual")
    ax_res.set_yscale("symlog", linthresh=1e-6)
    
    major_ticks = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]    
    ax_res.set_yticks(major_ticks)
    ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
    ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

fig.tight_layout()
fig.subplots_adjust(hspace=0.05, wspace=0.2)

plt.show()

# Check against analytical results varying scale factors

### Define events with different scale factor combinations $(a(x), a_0)$

In [ ]:
quadri_vectors = jnp.array([             # quadri-vectors defined at the center of the volume considering different scale factors
    [0.999, L_box/2, L_box/2, L_box/2],
    [0.5,   L_box/2, L_box/2, L_box/2],
    [0.5,   L_box/2, L_box/2, L_box/2],
    [1.0,   L_box/2, L_box/2, L_box/2],
    [1.0,   L_box/2, L_box/2, L_box/2]
])

scale_factor_obs = jnp.array([           # scale factors wrt the observer
    1.0,
    0.5,
    0.501,
    1.003,
    0.999
]) 

LoS_directions = jnp.zeros((quadri_vectors.shape[0], 3))
LoS_directions = LoS_directions.at[..., 0].set(1.0)      # line-of-sight directions corresponding to N_sk skewers pointing towards the z-direction, shape (N_sk, N_ppsk, 3)

### Compute line profiles

In [ ]:
phi_ns = line_profile.compute_line_profiles(
    quadri_vectors,
    LoS_directions,
    scale_factor_obs,
    wavelengths,
    line_bundle,
    fields,
    voigt_implementation_mode = "ExoJAX"
)
print("Shape of phi_ns", phi_ns.shape, ". Expected shape (N_lines, *batch_shape, N_lambda) = (",line_bundle.n_lines,",",quadri_vectors.shape[:-1],",",len(wavelengths),")")

### Analytic benchmark

In [ ]:
gamma_GHz = line_bundle.A21_GHz / (4.0 * jnp.pi)

lambdas_A_prime = line_profile.compute_lambda_prime(
    wavelengths,
    scale_factor_obs=scale_factor_obs,
    scale_factor=quadri_vectors[:, 0],
    LoS_velocity=jnp.zeros((line_bundle.n_lines, len(scale_factor_obs)))
) # (N_lines, N_points, N_lambda)

lambda0_A = jnp.asarray(line_bundle.lambda0_A)  # (N_lines,)
nu_GHz = ((1.0 / lambdas_A_prime) - (1.0 / lambda0_A[:, None, None])) * phys.c * 10.0

phi_analytic = Lorentzian(nu_GHz, gamma_GHz[:, None, None]) # (N_lines, N_points, N_lambda)

### Validation plot

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)           # (N_lambda,)
phi_np = np.asarray(phi_ns)                    # (N_lines, N_points, N_lambda)
phi_analytic = np.asarray(phi_analytic)        # (N_lines, N_points, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:phi_np.shape[1]])

# Zoom around each line center [Angstrom]
delta_lambda_A = 10.   # adjust to taste
N_lines = line_bundle.n_lines

fig, axes = plt.subplots(
    nrows=2, ncols=N_lines, 
    figsize=(14, 8), 
    sharex='col',
    sharey='row',
    gridspec_kw=dict(height_ratios=[2, 1])
)

if N_lines == 1:
    axes = axes.reshape(2, 1)

for i_sp, (ax_main, ax_res, line) in enumerate(zip(axes[0], axes[1], line_bundle.lines)):

    lambda0 = line.lambda0_A
    
    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]

    _lambda0 = line.lambda0_A
    ax_main.axvline(_lambda0, color='k', linestyle="-", linewidth=3.5)

    for i_point in range(phi_np.shape[1]):
        
        _lambda0 = line.lambda0_A * scale_factor_obs[i_point] / quadri_vectors[i_point, 0]
        ax_main.scatter(
            np.array([_lambda0]), 
            np.array([4./line_bundle.lines[i_sp].A21_GHz]), 
            marker="x", color=colors[i_point], s=100, alpha=1.0
        )
        ax_main.axvline(_lambda0, color=colors[i_point], linestyle="-.", linewidth=1.5)
        ax_res.axvline(_lambda0, color=colors[i_point], linestyle="-.", linewidth=1.5)

        ax_main.plot(lam_win, phi_np[i_sp, i_point, mask], color=colors[i_point], linestyle='-', linewidth=2.0, label='SkewerJAX')
        ax_main.plot(lam_win, phi_analytic[i_sp, i_point, mask], color=colors[i_point], linestyle=':', linewidth=4.0, alpha=0.6, label='Analytical')
        
        residual = phi_np[i_sp, i_point, mask] / phi_analytic[i_sp, i_point, mask] - 1
        
        ax_res.plot(lam_win, residual, color=colors[i_point], linestyle='-', linewidth=0.1, alpha=0.3)

    ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
    ax_res.fill_between(lam_win, -1e-6, 1e-6, color="grey", alpha=0.2)
    ax_res.hlines(1e-6, xmin=lam_win[0], xmax=lam_win[-1], color="k", linewidth=1, linestyle='--', alpha=0.5)
    ax_res.hlines(-1e-6, xmin=lam_win[0], xmax=lam_win[-1], color="k", linewidth=1, linestyle='--', alpha=0.5)
    
    ax_res.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax_res.set_ylabel("residual")
    ax_res.set_yscale("symlog", linthresh=1e-6)
    
    major_ticks = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]    
    ax_res.set_yticks(major_ticks)
    ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
    ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    
    ax_res.set_ylim(-1e-2, 1e-2)

    ax_main.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax_main.set_yscale("log")
    ax_main.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

# Legend different scale factor ratios (colors)
handles = []
for i_point in range(phi_np.shape[1]):
    _a = str(quadri_vectors[i_point, 0])
    _a0 = str(scale_factor_obs[i_point])
    lbl = r"$a/a_0 = $" + _a + r"$/$" + _a0
    handles.append(Line2D([0], [0], color=colors[i_point], lw=4, linestyle="-", label=lbl))
axes[0, 0].legend(handles=handles, loc="upper left", fontsize=16, ncols=1)

fig.tight_layout()
fig.subplots_adjust(hspace=0.05, wspace=0.2)

plt.show()

# Check variations with temperature and LoS velocity

### Define the fields to sample

In [ ]:
L_box = 1000.0 # UNITS: [Mpc]

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)), # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=-300.0, v0_max=300.0, L_box=L_box)),  # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                            # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))                           # UNITS: [GHz]
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)), # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=-300.0, v0_max=300.0, L_box=L_box)),  # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                            # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))                           # UNITS: [GHz]
    }
})

### Define wavelength range and lines to study

In [ ]:
wavelengths = jnp.linspace(970, 1280, 2**18)  # in Angstroms
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

### Define points to evaluate

In [ ]:
N_sk = 7
N_ppsk = 5
N_grid = 64

points = aux.make_plane_points(L_box, 'y', L_box/2, N_sk, N_ppsk)  # spatial points corresponding to N_sk skewers of length N_ppsk along the y-plane with y = L_box/2, shape (N_sk, N_ppsk, 3)
scale_factors = jnp.ones(points.shape[0:-1])[..., None]
quadri_vectors = jnp.concatenate([scale_factors, points], axis=-1) # quadri-vectors corresponding to N_sk skewers considering constant scale factors = 1.0, shape (N_sk, N_ppsk, 4)

LoS_directions = jnp.zeros_like(points)
LoS_directions = LoS_directions.at[..., 0].set(1.0)                # line-of-sight directions corresponding to N_sk skewers pointing towards the x-direction, shape (N_sk, N_ppsk, 3)

scale_factor_obs = jnp.ones(quadri_vectors.shape[0:-1])            # scale factor wrt the observer, here we are assuming no cosmological expansion, a0=1

### Compute line profiles

In [ ]:
phi_ns = line_profile.compute_line_profiles(
    quadri_vectors,
    LoS_directions,
    scale_factor_obs,
    wavelengths,
    line_bundle,
    fields,
    voigt_implementation_mode = "ExoJAX",
)
print("Shape of phi_ns", phi_ns.shape, ". Expected shape (N_lines, *batch_shape, N_lambda) = (",line_bundle.n_lines,",",quadri_vectors.shape[:-1],",",len(wavelengths),")")

### Plot showing temperature variations

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)           # (N_lambda,)
phi_np = np.asarray(phi_ns)                    # (N_sp, N_sk, N_ppsk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_ppsk = N_ppsk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_sk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for temperatures (colors)
temp_handles = []
for i_sk in range(N_sk):
    _sampled_T = fields.temperature(quadri_vectors[i_sk])
    lbl = rf"$T = {_sampled_T[0, 0]:.2g}\,\mathrm{{K}}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=4, linestyle="-", label=lbl))
leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_T)


# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=line_bundle.n_lines)

fig.tight_layout()
plt.show()



# Zoom around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=line_bundle.n_lines, figsize=(14, 6), sharex=False, sharey=True)
if line_bundle.n_lines == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_sk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()

### Plot showing $v_{LoS}$ variations

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)                          # (N_lambda,)
phi_np = np.asarray(phi_ns)                                    # (N_sp, N_sk, N_ppsk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_sk = N_sk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_ppsk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and LoS velocities", fontsize=20)

# Legend for velocities (colors)
handles = []
for i_ppsk in range(N_ppsk):
    _sampled_fields = fields.sample(quadri_vectors[:, i_ppsk], LoS_directions[:, i_ppsk])
    _sampled_vLoS = _sampled_fields['HI']["LoS_velocity"][0]
    lbl = r"$v_\mathrm{{LOS}} = $" + str(round(_sampled_vLoS, 1)) + r" $\mathrm{km/s}$"
    handles.append(Line2D([0], [0], color=colors[i_ppsk], lw=4, linestyle="-", label=lbl))
leg = ax.legend(handles=handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg)


# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=line_bundle.n_lines)

fig.tight_layout()
plt.show()


# Zoom around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=line_bundle.n_lines, figsize=(14, 6), sharex=False, sharey=True)
if line_bundle.n_lines == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_ppsk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()